# Setup storage and shared tables

This notebook creates the shared storage and table setup for the workspace — everything that's identical regardless of which dataset gets onboarded, so it's created once here rather than duplicated in every dataset's setup notebook.

- Create the project catalog and shared bronze/silver/gold schemas.
- Create the shared landing and files volumes.
- Create the shared `bronze.ingestion_runs` table.
- Create the shared Silver tables (`silver.image_inventory`, `silver.leakage_groups`, `silver.rejected_records`) and their shared label columns.
- Leave dataset-specific paths and DDL (Bronze source metadata, any dataset-specific label axis) to the dataset setup notebooks.


In [ ]:
%run ./_setup_env


In [ ]:
from data_platform.layout import load_yaml_config
from data_platform.spark_io import ensure_silver_tables

STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
LANDING_ROOT = STORAGE_CONFIG["landing_root"]
STORAGE_ROOT = STORAGE_CONFIG["storage_root"]

CATALOG_NAME = "derm_showcase_project"

MEDALLION_LAYERS = ["bronze", "silver", "gold"]


In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"USE CATALOG {CATALOG_NAME}")

for layer in MEDALLION_LAYERS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{layer}")

spark.sql("CREATE VOLUME IF NOT EXISTS bronze.landing")
spark.sql("CREATE VOLUME IF NOT EXISTS bronze.files")

dbutils.fs.mkdirs(LANDING_ROOT)
dbutils.fs.mkdirs(f"{LANDING_ROOT.rstrip('/')}/archives")

display(spark.sql(f"SHOW VOLUMES IN {CATALOG_NAME}.bronze"))
display(spark.sql("SHOW SCHEMAS"))
display(dbutils.fs.ls(LANDING_ROOT))
display(dbutils.fs.ls(STORAGE_ROOT))

In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.ingestion_runs (
      ingestion_run_id STRING,
      dataset_name STRING,
      source_version STRING,
      started_at TIMESTAMP,
      finished_at TIMESTAMP,
      status STRING,
      records_seen INT,
      records_written INT
    )
    USING DELTA
    """
)

ensure_silver_tables(spark)

display(spark.sql("SHOW TABLES IN bronze"))
display(spark.sql("SHOW TABLES IN silver"))
display(spark.sql("DESCRIBE TABLE silver.image_inventory"))


## Storage conventions

- Landing volume convention: `archives/<source-archive>.zip`
- Bronze storage convention: `isic_2019/...`
- Silver storage convention: `silver/<dataset>/...`
- Gold storage convention: `gold/<product>/<version>/...`


## Dataset-specific setup

Dataset notebooks should create dataset-specific paths and dataset-specific tables.

Current example:

- `notebooks/isic_2019/05_setup_tables_and_folders.ipynb`
- `notebooks/isic_2019/10_bronze_ingest.ipynb`

That notebook should load dataset config, create dataset-specific DDL, and manage dataset-specific folder paths.